# Discretionary Portfolio Viewer


In [1]:
from datetime import datetime, timedelta
from pathlib import Path
import sys

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, clear_output, display


def _bootstrap_repo_path() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "tradingagents").exists() and (candidate / "tools").exists() and (candidate / "data").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return candidate
    raise FileNotFoundError("Could not find Active Portfolio repo root in current path or parents")


REPO_ROOT = _bootstrap_repo_path()
from activeportfolio.portfolio import DiscretionaryPortfolio

DEFAULT_STATE_PATH = REPO_ROOT / "eval_results" / "discretionary_portfolio" / "default_portfolio.json"
DEFAULT_DATA_ROOT = REPO_ROOT / "data" / "market"


def _normalize_symbol(raw: str) -> str:
    return str(raw).strip().upper()


def _as_of_datetime(value) -> datetime:
    if value is None:
        return datetime.utcnow()
    return datetime.combine(value, datetime.min.time())


def _load_portfolio(state_path: str) -> DiscretionaryPortfolio:
    return DiscretionaryPortfolio.load(state_path=str(state_path), data_root=str(DEFAULT_DATA_ROOT))


def _load_ticker_history(symbol: str, as_of: datetime) -> pd.DataFrame:
    symbol = _normalize_symbol(symbol)
    symbol_dir = DEFAULT_DATA_ROOT / symbol
    files = sorted(symbol_dir.glob("history_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No local history for {symbol}")

    df = pd.concat([pd.read_parquet(path) for path in files], ignore_index=True)
    if "Date" not in df.columns:
        raise ValueError(f"Missing Date column in history for {symbol}")

    price_col = "Adj Close" if "Adj Close" in df.columns else "Close"
    if price_col not in df.columns:
        raise ValueError(f"Missing price column for {symbol}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])

    one_year_ago = as_of - timedelta(days=365)
    df = df[(df["Date"] >= one_year_ago) & (df["Date"] <= as_of)]
    if df.empty:
        raise ValueError(f"No 1Y rows for {symbol}")

    return (
        df[["Date", price_col]]
        .rename(columns={price_col: "Close"})
        .sort_values("Date")
        .reset_index(drop=True)
    )


def _marker_color(action: str) -> str:
    return "#1f7a1f" if str(action).upper() == "BUY" else "#b22222"


def _marker_symbol(action: str) -> str:
    return "triangle-up" if str(action).upper() == "BUY" else "triangle-down"


def _activity_label(row: pd.Series) -> str:
    if "target_weight" in row and pd.notna(row["target_weight"]):
        return f"{row['action']} target={float(row['target_weight']):.4f}"
    return f"{row['action']} shares={float(row.get('shares', 0.0)):.4f}"


ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
state_path_input = widgets.Text(value=str(DEFAULT_STATE_PATH), description="State path", layout=widgets.Layout(width="52%"))
as_of_input = widgets.DatePicker(description="As of", value=datetime.now().date())
ticker_input = widgets.Text(value="AAPL", description="Ticker")
refresh_button = widgets.Button(description="Refresh", button_style="primary")
portfolio_out = widgets.Output()
chart_out = widgets.Output()


def render(_event=None):
    state = _load_portfolio(state_path_input.value)
    as_of = _as_of_datetime(as_of_input.value)
    symbol = _normalize_symbol(ticker_input.value)

    with portfolio_out:
        clear_output(wait=True)
        total, frame = state.current_valuation(as_of=as_of)

        display(Markdown(f"# Portfolio: {state.name}"))
        display(Markdown(f"- Cash: `{state.cash:.2f}`"))
        display(Markdown(f"- Total valuation: `{total:.2f}`"))

        if frame.empty:
            display(Markdown("No active holdings."))
        else:
            frame = frame.copy()
            frame["weight"] = frame["market_value"] / total if total > 0 else 0.0
            display(frame.reset_index(drop=True))

        activities = state.activity_log()
        if activities.empty:
            display(Markdown("No activities yet."))
        else:
            display(Markdown("### Recent activities"))
            display(activities.tail(30))

    with chart_out:
        clear_output(wait=True)
        try:
            history = _load_ticker_history(symbol, as_of)
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=history["Date"], y=history["Close"], mode="lines", name=f"{symbol} Close (1Y)"))

            acts = state.activity_log()
            if not acts.empty:
                symbol_acts = acts.loc[acts["symbol"] == symbol].copy()
                if not symbol_acts.empty:
                    symbol_acts["trade_date"] = pd.to_datetime(symbol_acts["trade_date"])
                    window = symbol_acts[(symbol_acts["trade_date"] >= as_of - timedelta(days=365)) & (symbol_acts["trade_date"] <= as_of)]
                    for _, row in window.iterrows():
                        action = str(row["action"]).upper()
                        trade_date = row["trade_date"]
                        try:
                            px = float(history.loc[history["Date"] <= trade_date, "Close"].iloc[-1])
                        except Exception:
                            px = history["Close"].iloc[-1]

                        fig.add_vline(
                            x=trade_date,
                            line=dict(color=_marker_color(action), width=1, dash="dot"),
                        )
                        fig.add_trace(
                            go.Scatter(
                                x=[trade_date],
                                y=[px],
                                mode="markers",
                                marker=dict(
                                    size=11,
                                    color=_marker_color(action),
                                    symbol=_marker_symbol(action),
                                ),
                                name=_activity_label(row),
                            )
                        )

            fig.update_layout(
                title=f"{symbol} last 1 year",
                template="plotly_white",
                xaxis_title="Date",
                yaxis_title="Price",
                height=520,
            )
            fig.show()
        except Exception as exc:
            display(Markdown(f"Unable to render chart: {exc}"))


refresh_button.on_click(render)
render()

display(
    widgets.VBox(
        [
            widgets.HBox([state_path_input, as_of_input, ticker_input, refresh_button]),
            portfolio_out,
            chart_out,
        ]
    )
)
